# Real Data Preparation

This notebook documents how the project moves from the original public dog-growth data source to smaller processed CSV files that can be used safely in the modelling notebooks.

The full original dataset is not stored in this repository. The project keeps the original public data as a local **raw dataset archive** when regeneration is needed, while the current notebooks use processed CSV samples from `data/processed/`.

The purpose of this notebook is to make the data workflow clear, reproducible, and suitable for GitHub review.

## Data Source

The real data source is the University of Liverpool DataCat dataset:

**Growth standard charts for monitoring bodyweight in dogs of different sizes - SUPPORTING DATA**

The dataset is documented in:

- `docs/real_data_source_notes.md`
- `docs/real_data_download_instructions.md`
- `docs/raw_dataset_archive_policy.md`
- `docs/data_preparation_plan.md`
- `DATA_SOURCES.md`

Important terminology:

```text
raw dataset archive = the original public dataset file distributed by the source
processed CSV sample = the smaller project-ready dataset used by the notebooks
```

The term **raw dataset archive** does not refer to project patch archives, clean checkpoint archives, or any temporary development ZIP files.

## Mathematical Formulation

This notebook is a data preparation notebook, so it does not train a final predictive model. Its mathematical role is to transform raw public records into a clean feature matrix that can be used by later notebooks.

### Input vector `X`

The raw dataset contains public dog growth records. After preparation, each row can be converted into a modelling vector such as:

```text
X = [visit_age_months, weight_kg, gender_encoded, average_adult_breed_weight_kg, bcs_features]
```

### Target `y`

The target depends on the downstream task:

```text
Regression:      y = weight_kg or expected_weight
Classification:  y = growth_status_binary
```

For this preparation stage, the notebook focuses on creating clean columns that future models can use.

### Model function `f(x)`

No final model is trained here. The important function is the data transformation pipeline:

```text
T(raw_records) -> clean_table -> X, y
```

### Loss function

No predictive loss is optimized in this notebook. Downstream notebooks will use:

```text
Regression:      MSE / MAE
Classification:  LogLoss / classification error
```

Here, correctness is checked through data-quality rules instead of a model loss.

### Metrics

The preparation step is checked with data-quality indicators:

```text
row count
column count
missing values
valid value ranges
usable target availability
```

### Interpretation

This notebook connects the project to a real public dog growth dataset and prepares it for mathematical modelling.

### Limitations

The public dataset is not a private Cane Corso veterinary dataset. It provides a realistic dog-growth foundation, while the Cane Corso framing is the project domain and product interpretation.


## Important Repository Rule

The original public dataset archive should not be committed directly to GitHub.

The repository keeps:

- documentation about the original source;
- lightweight source notes in `data/raw/`;
- smaller processed CSV samples in `data/processed/`.

This keeps the project transparent, lightweight, and reproducible.

In [1]:
from pathlib import Path
import zipfile
import pandas as pd


In [2]:
project_root = Path.cwd()
if not (project_root / 'data').exists() and (project_root.parent / 'data').exists():
    project_root = project_root.parent

raw_data_dir = project_root / 'data' / 'raw'
processed_data_dir = project_root / 'data' / 'processed'

expected_raw_archive = raw_data_dir / 'Final_Data_PLOS.zip'
processed_sample_path = processed_data_dir / 'dog_growth_public_sample.csv'

print('Raw data folder:', raw_data_dir)
print('Processed data folder:', processed_data_dir)
print('Expected raw dataset archive:', expected_raw_archive)
print('Processed sample path:', processed_sample_path)


Raw data folder: c:\Users\stana\Desktop\cane-corso-growth-intelligence-current\data\raw
Processed data folder: c:\Users\stana\Desktop\cane-corso-growth-intelligence-current\data\processed
Expected raw dataset archive: c:\Users\stana\Desktop\cane-corso-growth-intelligence-current\data\raw\Final_Data_PLOS.zip
Processed sample path: c:\Users\stana\Desktop\cane-corso-growth-intelligence-current\data\processed\dog_growth_public_sample.csv


## Check Whether the Raw Dataset Archive Exists Locally

This check should not fail the notebook.

If the file is missing, it only means that the original public dataset archive is not currently stored on this machine. That is acceptable because the project already includes processed CSV samples for the current experiments.

In [3]:
if expected_raw_archive.exists():
    print('Raw dataset archive found locally.')
else:
    print('Raw dataset archive is not available locally.')
    print('This is OK for normal notebook review because processed CSV samples are included.')
    print('Download instructions are documented in docs/real_data_download_instructions.md')


Raw dataset archive is not available locally.
This is OK for normal notebook review because processed CSV samples are included.
Download instructions are documented in docs/real_data_download_instructions.md


## Inspect Raw Dataset Archive

The original public dataset is distributed as a compressed archive. The archive itself is not stored in the repository because raw downloaded data should remain outside version control.

If the archive is available locally, this section lists its contents before processing. If it is not available, the notebook continues safely and uses the processed sample files included in the project.

This section is part of the data-source workflow, not a dependency on project patch files or clean ZIP checkpoints.

In [4]:
if expected_raw_archive.exists():
    with zipfile.ZipFile(expected_raw_archive, 'r') as archive_file:
        files = archive_file.namelist()
        print('Files inside raw dataset archive:')
        for file_name in files[:20]:
            print('-', file_name)
else:
    print('Skipping archive inspection because the raw dataset archive is not available locally.')
    print('Using processed CSV samples already included in data/processed/.')


Skipping archive inspection because the raw dataset archive is not available locally.
Using processed CSV samples already included in data/processed/.


## Cleaning Steps Used for the Processed Samples

When the raw dataset archive is available locally, the preparation process is:

1. Inspect available files and columns inside the raw dataset archive.
2. Select age and bodyweight-related columns.
3. Check missing values.
4. Check unrealistic values.
5. Rename columns for clarity.
6. Create smaller processed samples.
7. Save the processed samples in `data/processed/`.

The same workflow is implemented in:

```text
src/create_public_sample.py
src/create_classification_sample.py
```

## Processed Datasets Used by the Project

The current project uses these processed files:

```text
data/processed/dog_growth_public_sample.csv
data/processed/dog_growth_classification_sample.csv
```

These files are small enough to commit to GitHub and stable enough to use in the modelling notebooks.

## Current Status

The project currently has:

- a completed regression notebook using prototype data;
- real public data source documentation;
- a clear raw dataset archive policy;
- processed public dog-growth samples in `data/processed/`;
- classification notebooks using the processed classification sample.

The raw dataset archive is only needed if the processed samples need to be regenerated.